## UE CAPSIM - Introduction à l'IRM fonctionnelle

Ce lab est divisé en deux parties :
1. Généralités
2. Prétraitement

Ces deux premières parties sont destinées à vous faire manipuler quelques concepts essentiels sur les signaux IRMf, en utilisant la librairie [nilearn](https://nilearn.github.io/stable/index.html). 

Ensuite, nous détaillons quelques uns des concepts vus en cours par des exemples et des liens documentés sur le site de nilearn: 
- Le modèle de régression linéaire général (GLM)
- Analyse de connectivité fonctionnelle
- Apprentissage automatique

### 1. Généralités


Nous présentons ici une version résumée des possibilités de nilearn. 

Pour commencer, chargeons un jeu de données simple et examinons ses metadonnées. 

In [ ]:
from nilearn import datasets
import numpy as np
from nilearn import image

import matplotlib.pyplot as plt

# Charger le dataset d'imagerie auditive de SPM
data = datasets.fetch_spm_auditory()

# Charger les images anatomique et fonctionnelle
anat_img = image.load_img([data.anat])
func_img = image.load_img([data.func])

# Visualiser les métadonnées du dataset
# ceci est important pour comprendre les dimensions des images, la résolution spatiale et temporelle,
#  et d'autres caractéristiques clés qui influenceront les étapes de prétraitement et d'analyse ultérieures.
# Nous utilisons pour cela les attributs des objets Nifti1Image chargés avec nibabel (via nilearn) pour extraire ces informations.
# notamment avec la méthode get_zooms() pour obtenir la taille des voxels et le TR (repetition time) de l'image fonctionnelle.
print("=" * 50)
print("fMRI Dataset Metadata")
print("=" * 50)
print(f"\nAnatomical image shape: {anat_img.shape}")
print(f"Anatomical voxel size: {anat_img.header.get_zooms()[0]:.2f} x {anat_img.header.get_zooms()[1]:.2f} x {anat_img.header.get_zooms()[2]:.2f}")
print(f"\nFunctional image shape: {func_img.shape}")
print(f"Functional voxel size: {func_img.header.get_zooms()[0]:.2f} x {func_img.header.get_zooms()[1]:.2f} x {func_img.header.get_zooms()[2]:.2f}")
print(f"Repetition time (TR): {func_img.header.get_zooms()[3]:.2f} seconds")
print(f"Number of time points: {func_img.shape[3]}")


déduisez en la durée totale d'acquisition de cette session d'IRMf

In [1]:
## TODO: calculer la durée totale de l'acquisition fMRI


La cellule suivante montre comment utiliser les fonctions de visualisation de nilearn pour visualiser les données anatomiques et fonctionnelles. 

Les fonctions nilearn importantes : 
- plot_anat pour les données anatomiques (IRM structurelle T1)
- plot_epi pour les données IRMf

In [ ]:
# Display anatomical image


# Visualize anatomical and a functional volume
fig, axes = plt.subplots(2,1, figsize=(10, 12))


zval = 10
# Anatomical image
# plot using nilearn's plotting function
from nilearn import plotting
plotting.plot_anat(anat_img, title='Anatomical Image', axes=axes[0], colorbar=False)
axes[0].axis('off')

# Functional volume (first timepoint)
plotting.plot_epi(func_img.slicer[..., 0], title='Functional Volume (First Timepoint)', axes=axes[1],colorbar=False)
axes[1].axis('off')

plt.show()


La cellule suivante montre un exemple d'extraction du signal d'un voxel unique. 

In [ ]:

# Extract and plot time series of a single voxel
func_data = func_img.get_fdata()
voxel_i, voxel_j, voxel_k = 40, 50, 25
time_series = func_data[voxel_i, voxel_j, voxel_k, :]


## TODO : visualiser la série temporelle du signal BOLD pour un voxel donné (i, j, k) en fonction du temps (en volumes ou en secondes)
plt.figure(figsize=(12, 4))
plt.plot(time_series)
plt.xlabel('Time (volumes)')
plt.ylabel('Signal intensity')
plt.title(f'Time Series of Voxel at ({voxel_i}, {voxel_j}, {voxel_k})')
plt.grid(True, alpha=0.3)
plt.show()

Pour davantage de détails sur la manipulation et la visualisation, consultez la section User Guide du site de nilearn. 

En particulier, pour mieux comprendre les structures de données nilearn image, consultez [cette partie : understanding neuroimaging data](https://nilearn.github.io/stable/manipulating_images/input_output.html). 

Pour un tour d'horizon des possibilités de visualisation, voir [ici : Plotting Brain Images](https://nilearn.github.io/stable/plotting/index.html)

### 2 - Prétraitement


La plupart des étapes de prétraitement s'effectue **avant** nilearn, l'outil le plus populaire aujourd'hui étant [fmriprep](https://fmriprep.org/en/stable/) qui automatise les étapes de prétraitement minimal. Les étapes que l'on peut réaliser dans nilearn sont : 
- le filtrage spatial
- le filtrage temporel (ex : supprimer la dérive lente, en anglais **detrending** )
- effectuer une regression pour retirer l'influence de variables confondantes telles que l'estimation des mouvements de tête, ou le signal venant du CSF (**cerebro spinal fluid**, liquide encéphalorachidien)


Nous allons d'abord visualiser l'effet du filtrage spatial. 

Utilisez la fonction `smooth_img` du module `nilearn.image`, et visualisez l'effet du filtre avec différents paramètres. Consultez la documentation de nilearn. 

In [ ]:
from nilearn.image import smooth_img

## TODO : appliquer un lissage spatial à l'image fonctionnelle, comparez les résultats avant et après lissage, et avec différentes valeurs de FWHM (ex: 4mm, 6mm, 8mm)


Effet du filtrage temporel

Le filtrage temporel est effectué sur les séries temporelles de chaque voxel séparément. 
Afin de pouvoir traiter sur les voxels indépendemment, il nous faut définir un **masque** binaire permettant d'identifier quels voxels analysés. Ce masque est une image *mask_img* des mêmes dimensions que *func_img*, mais qui contient la valeur 1 pour les voxels à analyser, et 0 partout ailleurs. Typiquement, on vise à ce que le masque corresponde au cerveau (voire, à la matière grise uniquement). 

Le masque est un des résultats du prétraitement, mais il est également possible de l'estimer directement dans nilearn. 
On peut utiliser pour cela la fonction [compute_epi_mask](https://nilearn.github.io/stable/modules/generated/nilearn.masking.compute_epi_mask.html#nilearn.masking.compute_epi_mask)

In [ ]:
## estimation d'un masque à partir de func_img
## On peut utiliser pour cela la fonction compute_epi_mask
from nilearn.masking import compute_epi_mask

## TODO : calculer un masque binaire à partir de l'image fonctionnelle, 
# ## visualiser ce masque et discuter de son importance pour les étapes d'analyse ultérieures 

On visualise ensuite ce masque avec la fonction [plot_roi](https://nilearn.github.io/stable/modules/generated/nilearn.plotting.plot_roi.html#nilearn.plotting.plot_roi) , en mettant en image de fond un des volumes d'IRMf en utilisant `func_img.slicer[...,0]` comme précédemment. 
Il faut ici désactiver la colorbar car les valeurs ne sont pas pertinentes

In [ ]:
## TODO


Nous pouvons utiliser la fonction [apply_mask](https://nilearn.github.io/stable/modules/generated/nilearn.masking.apply_mask.html#nilearn.masking.apply_mask) afin de récupérer uniquement les valeurs d'un volume d'IRMf correspondant au masque

Combien de voxels sont effectivement à 1 dans ce masque ? C'est à dire, combien de voxels peuvent être analysés avec ce masque ? 

In [ ]:

from nilearn.masking import apply_mask
# TODO : appliquer le masque à l'image fonctionnelle pour extraire les séries temporelles des voxels à l'intérieur du masque, 
# et visualiser la forme de la matrice résultante (nombre de voxels x nombre de temps)




In [ ]:
## TODO : visualiser la matrice des séries temporelles de tous les voxels à l'intérieur du masque (nombre de voxels x nombre de temps) *
## avec la fonction plot_carpet de nilearn.plotting
## La fonction plot_carpet du module nilearn.plotting permet de faire une visualisation des séries temporelles de l'ensemble des voxels
## On peut aussi donner en argument le masque (par défaut, un masque est estimé à partir des données)


Montrez l'effet du filtrage temporel en utilisant la fonction `butterworth`du module `nilearn.signal`, qui applique un [filtre de Butterworth](https://fr.wikipedia.org/wiki/Filtre_de_Butterworth).

Appliquez différents types de filtre (passe-bas, passe-haut) et observez l'effet sur le signal BOLD. 

Attention : utilisez l'argument `copy=True` afin de pouvoir tester facilement plusieurs valeurs, sinon les données originales seront écrasées. 

In [ ]:
## TODO : appliquer un filtre temporel (ex: Butterworth) pour ne garder que les fréquences d'intérêt (ex: 0.01 - 0.1 Hz) 
## sur les séries temporelles des voxels à l'intérieur du masque,

from nilearn.signal import butterworth


Visualisez l'effet du filtre sur un voxel quelconque

In [ ]:
## TODO : visualiser les données filtrées et les données originales pour un voxel donné


On peut ensuite remettre les données masquées sur une structure nilearn image, afin de pouvoir utiliser les fonctions de visualisation
Pour cela, utilisez la fonction `unmask` du module `nilearn.masking`

In [ ]:
## TODO 


Enfin, une des étapes les plus importantes du prétraitement consiste à enlever les variables confondantes estimées, telles que le mouvement de la tête, ou le signal provenant de sources telles que la matière blanche. Pour cela, nous allons charger un sujet d'un autre jeu de données pour visualiser ces variables confondantes et retirer leur influence. 


In [ ]:
# Charger un autre dataset pour voir les confounds
from nilearn import datasets
data_dev = datasets.fetch_development_fmri(n_subjects=1)
func_img = image.load_img(data_dev.func[0])
print(func_img.shape)

# Les confounds sont des variables qui peuvent influencer les données fMRI, telles que les mouvements de la tête, les fluctuations physiologiques, etc. 
# Ils sont souvent utilisés pour nettoyer les données avant l'analyse.
# Ils sont ici stockés dans un fichier TSV (tab-separated values) et peuvent être chargés avec la fonction pandas.read_csv
import pandas as pd

confounds = pd.read_csv(data_dev.confounds[0], sep='\t')
print(confounds.columns) ## titre des colonnes
print(confounds.head()) ## affiche les premières lignes du dataframe des confounds


In [ ]:
# TODO Visualiser les confounds de mouvement (translation en x, y, z) 


A étudier :  nilearn dispose de fonctions pour retirer l'influence des confounds sur les séries temporelles. 
Notamment, la fonction [`clean` du module `nilearn.signal`](https://nilearn.github.io/stable/modules/generated/nilearn.signal.clean.html#nilearn.signal.clean) permet de combiner l'application de filtres et la regression des confounds. 

### Le modèle linéaire général (GLM)

Nous avons en cours qu'une méthode courante consiste à modéliser la réponse BOLD en utilisant la fonction de réponse hémodynamique canonique (HRF). 
Une expérience d'IRMf peut ensuite être modélisée comme une suite d'événements ayant des temps de démarrage (onset) et des durées, ainsi que des variables confondantes. On construit ainsi une "design matrix" qui contient des régresseurs (événements, variables confondantes) en colonnes, et le temps en ligne. Cette matrice est ensuite utilisée pour apprendre un modèle de régression linéaire pour chaque voxel. 

Nous montrons ici quelques aspects clés de cette modélisation. 

Nilearn intègre différents modèles de HRF, nous montrons ici le modèle de Glover. 

L'axe des x comporte une unité arbitraire, car la fonction permettant de générer la HRF effectue une expansion temporelle. 

In [ ]:
## plot la HRF canonique utilisée pour la convolution
from nilearn.glm.first_level import glover_hrf
glover_hrf_values = glover_hrf(t_r=1.0)
plt.figure(figsize=(8, 4))
plt.plot(glover_hrf_values)
plt.title('Glover HRF')
plt.xlabel('Time (aribtrary units)')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)
plt.show()

Cette HRF sera convoluée avec les regresseurs. 

Vous pouvez voir dans [cet exemple](https://nilearn.github.io/stable/auto_examples/04_glm_first_level/plot_hrf.html) la réponse modélisée à trois différente HRF, ainsi que leur dérivées.

Dans [cet exemple](https://nilearn.github.io/stable/auto_examples/04_glm_first_level/plot_predictions_residuals.html), vous pouvez voir un exemple de modélisation sur des données réelles, comparant les signaux précédents aux réponses BOLD réelles. 

Un exemple complet d'analyse sur un seul sujet, une seule session, est disponible [ici](https://nilearn.github.io/stable/auto_examples/00_tutorials/plot_single_subject_single_run.html). Cet exemple comporte : 
- la modélisation de l'expérience par ces événements
- la génération de la design matrice
- l'estimation du modèle de régression
- la visualisation et le seuillage statistique des résultats

### Connectivité fonctionnelle

La connectivité fonctionnelle est typiquement estimée entre régions du cerveau, ou entre "réseaux", et non directement entre les voxels, ceux-ci étant beaucoup trop nombreux. Pour cela, la première étape consiste à effectuer une **parcellation** des données IRMf en utilisant un atlas de régions / réseaux. 

Il existe deux types d'atlas :
- ceux basés sur des "hard labels", comportant des régions disjointes : chaque voxel appartient à une seule et unique région,
- les atlas"probabilistes", ou chaque voxel dispose d'une probabilité d'appartenir à chaque région. 

Nilearn dispose d'objets de **masquage** permettant de parceller les données à partir des structure nilearn image. 
Les données masquées sont des numpy array de dimension $T \times N_{roi}$ ou $T$ est le nombre de volumes, et $N_{roi}$ est le nombre de régions. 


Nous vous demandons d'étudier cette [page du user guide de nilearn](https://nilearn.github.io/stable/connectivity/functional_connectomes.html), qui explique comment effectuer la parcellation des données et estimer la matrice de connectivité fonctionnelle. 

Vous pouvez ensuite regarder [cet exemple](https://nilearn.github.io/stable/auto_examples/03_connectivity/plot_atlas_comparison.html) effectuant une comparaison de différents atlas. 


### Apprentissage automatique

Nous vous suggérons de faire [cet exemple](https://nilearn.github.io/stable/auto_examples/00_tutorials/plot_decoding_tutorial.html) qui montre comment décoder des catégories visuelles. 

Enfin, vous pouvez voir comment classifier des matrices de connectivité fonctionnelle avec [cet exemple](https://nilearn.github.io/stable/auto_examples/03_connectivity/plot_group_level_connectivity.html).

Vous pouvez également explorer les différents exemples de la partie decoding. 
